# BFCL V4 — P06 smoke test (official harness)

Purpose: **actually run**, in a user-controlled GPU Colab session, 3 indexed families across base / missing function / missing parameter with a supported 4B model. This is a **smoke test**, not scientific evidence for a published result. This notebook was authored but **has not been run**. GPU must be enabled in Colab. Notebook pins the official Gorilla source commit; model revision is reported from resolved Hugging Face download information and should be frozen for full experiments. Data and model may require a substantial download.


In [ ]:
import subprocess, sys, os, json, pathlib, datetime, platform
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat())
print('Python', sys.version, 'Platform',platform.platform())
subprocess.run(['nvidia-smi'],check=True)



In [ ]:
from pathlib import Path
root=Path('/content/gorilla')
if not root.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ShishirPatil/gorilla.git',str(root)],check=True)
commit='6ea57973c7a6097fd7c5915698c54c17c5b1b6c8'
subprocess.run(['git','-C',str(root),'checkout',commit],check=True)
work=root/'berkeley-function-call-leaderboard'
print('Commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-e',str(work)+'[oss_eval_vllm]'],check=True)



In [ ]:
config={'multi_turn_base':['multi_turn_base_1'], 'multi_turn_miss_func':['multi_turn_miss_func_1'],'multi_turn_miss_param':['multi_turn_miss_param_1']}
(work/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
print('Selected smoke-test IDs:',config)
print('Do not extrapolate results from one family.')



In [ ]:
model='Qwen/Qwen3-4B-Instruct-2507-FC'
# Model ID is explicitly listed by the pinned BFCL SUPPORTED_MODELS.md as self-hosted FC.
# Requires compatible GPU and available Colab compute. May fail on constrained runtimes.
os.environ['BFCL_PROJECT_ROOT']=str(work)
cmd=['bfcl','generate','--model',model,'--run-ids','--backend','vllm','--num-gpus','1','--gpu-memory-utilization','0.85']
print('Running:', ' '.join(cmd))
subprocess.run(cmd,cwd=work,env=os.environ.copy(),check=True)



In [ ]:
categories='multi_turn_base,multi_turn_miss_func,multi_turn_miss_param'
cmd=['bfcl','evaluate','--model',model,'--test-category',categories,'--partial-eval']
print('Running:', ' '.join(cmd))
subprocess.run(cmd,cwd=work,env=os.environ.copy(),check=True)



In [ ]:
from google.colab import files
import shutil, hashlib
bundle=Path('/content/P06_BFCL_Smoke_Outputs')
bundle.mkdir(exist_ok=True)
for sub in ('result','score'):
    src=work/sub
    if src.exists():
        for path in src.rglob('*'):
            if path.is_file():
                dest=bundle/sub/path.relative_to(src)
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,dest)
manifest={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'bfcl_commit':commit,'model_id':model,'selected_ids':config,'note':'Smoke test, not complete study','files':{}}
for p in bundle.rglob('*'):
    if p.is_file():manifest['files'][str(p.relative_to(bundle))]=hashlib.sha256(p.read_bytes()).hexdigest()
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive('/content/P06_BFCL_Smoke_Outputs','zip',bundle)
print('Artifacts:',archive,'files:',len(manifest['files']))
files.download(archive)

